# 03 — Prompt Playground

The core iteration notebook. Load a transcript fixture, select a time window,
test prompts against the Apple Foundation Model, and see results.

This notebook is **parameterized** — Claude Code can drive it via papermill.

In [55]:
# Parameters (papermill-compatible)
book_slug = "hp-book1-ss"
chapter_index = 1  # "The Boy Who Lived"
position_seconds = -1  # -1 = end of chapter
window_minutes = 5.0
prompt_variant = "current_mini_recap_default"  # See lib/prompts.py for all variants

In [56]:
import sys, os
# Ensure lib/ is importable regardless of CWD
_eval_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if _eval_root not in sys.path:
    sys.path.insert(0, _eval_root)

from pathlib import Path
from lib.transcript import (
    load_fixture, load_chapters, format_transcript,
    format_transcript_display, select_window
)
from lib.prompts import get_prompt, build_combined_prompt, PROMPTS
from lib.token_budget import estimate_tokens, allocate_budget
from lib.fm_client import check_availability, call_model_timed

## Check FM availability

In [57]:
available, reason = check_availability()
print(f"FM Available: {available}")
print(f"Reason: {reason}")
assert available, f"Apple FM not available: {reason}"

FM Available: True
Reason: Available


## Load transcript and select window

In [58]:
# Resolve fixture dir robustly (works from notebook UI and papermill)
_nb_dir = Path("__file__").resolve().parent if Path("__file__").exists() else Path(".").resolve()
if _nb_dir.name == "notebooks":
    fixture_dir = _nb_dir.parent / "fixtures" / book_slug
else:
    fixture_dir = _nb_dir / "fixtures" / book_slug
if not fixture_dir.exists():
    # Fallback: try relative
    fixture_dir = Path(f"../fixtures/{book_slug}")

chapters = load_chapters(fixture_dir / "chapters.json")
ch = chapters[chapter_index]
print(f"Chapter {ch.index}: {ch.title}")

all_sentences = load_fixture(fixture_dir / f"transcript_ch{chapter_index:02d}.json")
print(f"Loaded {len(all_sentences)} sentences")

# Determine position
pos = position_seconds if position_seconds >= 0 else all_sentences[-1].end_time
print(f"\nSimulating playback position: {pos:.0f}s ({pos/60:.1f} min)")
print(f"Window: last {window_minutes} minutes")

window = select_window(all_sentences, position=pos, window_minutes=window_minutes)
print(f"Selected {len(window)} sentences")

Chapter 1: Chapter One - The Boy Who Lived
Loaded 391 sentences

Simulating playback position: 1948s (32.5 min)
Window: last 5.0 minutes
Selected 58 sentences


## Token budget analysis

In [59]:
from lib.prompts import PROMPTS

system, user = PROMPTS[prompt_variant]
# Instructions = system + user template without transcript
instructions = system + "\n\n" + user.replace("{transcript}", "")

budget = allocate_budget(instructions, window)

print(f"Token Budget Analysis")
print(f"{'='*40}")
print(f"Context limit:      {budget.total_limit}")
print(f"Instruction tokens: {budget.instruction_tokens}")
print(f"Transcript tokens:  {budget.transcript_tokens}")
print(f"Response reserve:   {budget.response_reserve}")
print(f"Safety margin:      {budget.safety_margin}")
print(f"Total used:         {budget.total_used}")
print(f"Headroom:           {budget.total_limit - budget.total_used - budget.response_reserve - budget.safety_margin}")
print(f"")
print(f"Sentences included: {budget.sentences_included}")
print(f"Sentences dropped:  {budget.sentences_dropped}")

Token Budget Analysis
Context limit:      4096
Instruction tokens: 132
Transcript tokens:  902
Response reserve:   800
Safety margin:      100
Total used:         1034
Headroom:           2162

Sentences included: 58
Sentences dropped:  0


## Transcript preview (what the model will see)

In [60]:
included = budget.included_sentences
print(f"Transcript being sent ({len(included)} sentences):\n")
print(format_transcript_display(included))

Transcript being sent (58 sentences):

[27:30] In his vast muscular arms, he was holding a bundle of blankets.
[27:35] Had greed, said Dumbledore, sounding relieved.
[27:39] At last. Where did you get that motorbike?
[27:43] Borrowed it, Professor Dumbledore, sir, said the giant, climbing carefully off the motorbike as he spoke.
[27:50] Young, serious black lent it me.
[27:53] I've got him, sir.
[27:55] No problems with that?
[27:56] No, sir. House was almost destroyed, but I got him out all right before the muggles started swarming around.
[28:02] He fell asleep as we was flying over Bristol.
[28:06] Dumbledore and Professor McGonigal bent forward over the bundle of blankets.
[28:11] Inside, just visible, was a baby boy, fast asleep.
[28:18] Under a tuft of jet-black hair over his forehead, they could see a curiously shaped cut, like a bolt of lightning.
[28:27] Is that weird?
[28:28] Yes. He'll have that scar forever.
[28:32] Couldn't you do something about it, Dumbledore?
[28:35] Ev

## Send prompt to Foundation Model

In [61]:
transcript_text = format_transcript(included)
combined = get_prompt(prompt_variant, transcript_text)

print(f"Prompt variant: {prompt_variant}")
print(f"Combined prompt tokens: ~{estimate_tokens(combined)}")
print(f"\nCalling Foundation Model...\n")

response, latency = await call_model_timed(combined)

print(f"Response ({latency:.1f}s, ~{estimate_tokens(response)} tokens):")
print(f"{'='*60}")
print(response)
print(f"{'='*60}")

Prompt variant: current_mini_recap_default
Combined prompt tokens: ~1034

Calling Foundation Model...

Response (2.5s, ~65 tokens):
In the last part of the story, Professor Dumbledore arrives at the Dursleys' doorstep with baby Harry Potter wrapped in blankets, leaving a letter inside. Dumbledore then vanishes into thin air, leaving Harry in the care of his aunt and uncle. As the scene ends, Harry remains asleep, unaware of his extraordinary destiny.


## Grounding check

In [62]:
from lib.evaluate import check_grounding

grounding = check_grounding(response, transcript_text)

print("Grounding Analysis:")
for g in grounding:
    status = "✓" if g["grounded"] else "✗"
    print(f"  {status} [{g['best_match_ratio']:.2f}] {g['sentence'][:80]}")

Grounding Analysis:
  ✗ [0.10] In the last part of the story, Professor Dumbledore arrives at the Dursleys' doo


## Custom prompt (edit and re-run)

Edit the prompt below to iterate. When you find something good, copy it back to `lib/prompts.py`.

In [74]:
# Edit this prompt and re-run the cell to iterate
custom_system = """
You are a voice audiobook assistant. 
Give a detailed, natural spoken recap of what the listener missed and nothing else, limited to 3-5 sentences.
""".strip()

custom_user = """
{transcript}
""".strip()

custom_combined = build_combined_prompt(custom_system, custom_user, transcript_text)
print(f"Custom prompt tokens: ~{estimate_tokens(custom_combined)}")
print(f"\nCalling Foundation Model...\n")

custom_response, custom_latency = await call_model_timed(custom_combined)

print(f"Response ({custom_latency:.1f}s):")
print(custom_response)

Custom prompt tokens: ~935

Calling Foundation Model...

Response (3.1s):
Dumbledore, McGonagall, and Hagrid have just delivered Harry Potter to the Dursleys, placing him on their doorstep with a letter tucked inside his blanket. As they depart, Hagrid is visibly upset, mourning the loss of Lily and James Potter, and McGonagall is silent, perhaps reflecting on the gravity of the situation. Dumbledore then vanishes with a flick of his cloak, leaving Privet Drive eerily quiet, oblivious to the magical significance of Harry's arrival. Unbeknownst to Harry, this marks the beginning of his journey and the world's awareness of him as the "boy who lived."


## Token limit calibration

Run this once to measure the actual context window boundary. Takes ~1 minute.

In [46]:
from lib.token_budget import calibrate_context_limit
from lib.fm_client import call_model

print("Calibrating context limit (this takes a minute)...\n")
calibration = await calibrate_context_limit(call_model)

print(f"Results:")
for k, v in calibration.items():
    print(f"  {k}: {v}")

Calibrating context limit (this takes a minute)...

Results:
  last_success_tokens: 1005
  first_failure_tokens: 1105
  estimated_correction_factor: 4.075621890547263
  effective_tiktoken_limit: 1005
